In [1]:
import pandas as pd
from rapidfuzz import fuzz, process
from deep_translator import GoogleTranslator
import re


In [2]:
# load full metadata
df = pd.read_excel('merged_journals_metadata.xlsx')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9434 entries, 0 to 9433
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        9434 non-null   object
 1   creator      9420 non-null   object
 2   publisher    9434 non-null   object
 3   date         9434 non-null   object
 4   type         9434 non-null   object
 5   format       9362 non-null   object
 6   identifier   9434 non-null   object
 7   source       9434 non-null   object
 8   language     9429 non-null   object
 9   relation     9370 non-null   object
 10  rights       7314 non-null   object
 11  oai_id       9434 non-null   object
 12  set_specs    9434 non-null   object
 13  set_names    9434 non-null   object
 14  subject      4812 non-null   object
 15  description  7437 non-null   object
 16  file         9434 non-null   object
dtypes: object(17)
memory usage: 1.2+ MB


In [4]:
keywords = (df['subject']
                .dropna()
                .str.replace('; ', ' | ', regex=False)
                .str.replace('；', ' | ', regex=False)
                .str.replace(', ', ' | ', regex=False)
                .str.replace('，', ' | ', regex=False)
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .dropna()
                .str.strip('| .,;')
                .str.lower())

keywords_stats_multilang = keywords.value_counts()

In [ ]:
unique_keywords = keywords.str.lower().unique().tolist()
unique_keywords = [re.sub(r'[„”«»“”]', '"', k) for k in unique_keywords]
unique_keywords = list(set(unique_keywords))
with open('unique_keywords.txt', 'w', encoding='utf-8') as txt:
    txt.writelines(line + '\n' for line in unique_keywords)
with open('to_translate.txt', 'w', encoding='utf-8') as txt:
    txt.writelines(line + '\n' for line in unique_keywords)

In [ ]:
translator = GoogleTranslator(source='auto', target='en')
with open('to_translate.txt', 'r', encoding='utf-8') as txt:
    to_translate = [e.strip() for e in txt.readlines()]
batch_size = 100
while to_translate: 
    batch = to_translate[:batch_size] 
    translated_batch = translator.translate_batch(batch)
    del to_translate[:batch_size]
    translated_merged = result = [f"{x.strip()} [--translated-->] {y.strip()}" for x, y in zip(batch, translated_batch)]
    with open('translated_keywords.txt', 'a', encoding='utf-8') as txt:
        txt.writelines(line + '\n' for line in translated_merged)
    with open('to_translate.txt', 'w', encoding='utf-8') as txt:
        txt.writelines(line + '\n' for line in to_translate)
    print('remaining to be translated -- ', len(to_translate))    

In [5]:
with open('translated_keywords.txt', 'r', encoding='utf-8') as txt:
    translated_keywords = [e.strip() for e in txt.readlines()]
translated_keywords = [e.split(' [--translated-->] ')[1] for e in translated_keywords]

with open('unique_keywords.txt', 'r', encoding='utf-8') as txt:
    unique_keywords = [e.strip() for e in txt.readlines()]
    
translation_map = dict(zip(unique_keywords, translated_keywords))

keywords_translated = keywords.map(translation_map)

keywords_stats_en = keywords_translated.value_counts()

keywords_df = pd.DataFrame({
    "original": keywords,
    "translated": keywords_translated,
})

keywords_df['translated'] = keywords_df['translated'].str.lower()
keywords_df = (
    keywords_df.dropna(subset=['translated'])
      .reset_index()
      .groupby(['index', 'translated'])
      .agg(originals=('original', lambda x: set(x)))
      .reset_index()
)

keywords_grouped = (
    keywords_df
    .groupby('translated')
    .agg(
        total_count=('translated', 'count'),
        originals_global=('originals', lambda x: list({o for s in x for o in s}))  
    )
    .sort_values(by='total_count', ascending=False)
    .reset_index()
)
keywords_grouped.head(10)

,translated,total_count,originals_global
0,translation,780,"[traducción, tradutoloxía, tradumàtica, perevo..."
1,audiovisual translation,230,"[tłumaczenie audiowizualne, 视听翻译, audiovisual ..."
2,literary translation,223,"[tradução literária, literaturübersetzen, trad..."
3,machine translation,116,"[traducción automática, tłumaczenie maszynowe,..."
4,subtitling,112,"[sous-titrage, legendagem, subtitulació, subti..."
5,translation studies,98,"[estudos da tradução, translatoryka, translato..."
6,audio description,90,"[audio description, audiodeskrypcja, 口述影像, aud..."
7,dubbing,89,"[dubbing, dobraxe, dublagem, doppiaggio, doubl..."
8,legal translation,85,"[traducció jurídica, traducción legal, rechtsü..."
9,translator training,79,"[formação de tradutores, formación del traduct..."


In [6]:
# authors stats
authors_stats = (df['creator']
                .dropna()
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .dropna()
                .str.strip())
authors_stats = pd.DataFrame(authors_stats)
affiliations_to_replace = ['(Chulalongkorn University, THAILAND), and ',
'(University of Manchester, UNITED KINGDOM), ',
'(Yarmouk University), ',
'(Jerash University), ',
'(Yarmouk University, JORDAN), ',
'(Beihang University, CHINA), ',
'(University of Extremadura, SPAIN), ',
'(University of Granada, SPAIN), ',
'(Universitat de València, SPAIN), ',
'(Leipzig University, GERMANY), ',
'(Guizhou Normal University, CHINA), ',
'(University Putra Malaysia, MALAYSIA; University of Newcastle, AUSTRALIA), ',
'(University Putra Malaysia, MALAYSIA), and ',
'(The John Paul II Catholic University of Lublin, POLAND), ',
'(Minia University, EGYPT), ',
'(University College London, UNITED KINGDOM), ',
'(Kent State University, UNITED STATES OF AMERICA), ',
'(The University of Western Australia, AUSTRALIA), ',
'(Queen’s University Belfast, UNITED KINGDOM), ',
'(National Police Academy, Ankara, TÜRKIYE), ',
'(Moscow City Teacher Training University, RUSSIA), ',
'(Universidad de Granada, SPAIN), ',
'(Hamad Bin Khalifa University, QATAR), ',
'(Leipzig University, GERMANY), and ',
'(University of Zadar/University of Applied Sciences Velika Gorica, CROATIA), ',
'(Kent State University, USA), ',
'(Hong Kong Baptist University, HONG KONG SAR), ',
'(Northwest Normal University/City University of Hong Kong), ',
'(City University of Hong Kong, HONG KONG SAR), ',
'(Politeknik El Bajo Commodus, INDONESIA), ',
'(Independent researcher, HONG KONG SAR), ',
', Universidad de Córdoba',]

for frag in affiliations_to_replace:
    authors_stats['creator'] = authors_stats['creator'].str.replace(frag, "", regex=False)

authors_field_to_drop = ['Tradução, Cadernos de',
'de Tradução, Cadernos',
'New Voices Editorial Team',
'Clina, Secretaría de Redacción',
'de redacción, Consello',
'Tradução, Cadernos',
'Notícias, Transfer',
'Autores, Varios',
'Transfer',
'UB, Transfer',
'Sendebar, Revista',
'Reviews, Book',
'Varios, Varios',
'Editorial, Equipo',
'de Traducción, Estudios',
'anonimo, anonimo',
'Equipo Editorial',
'Clina, Secretaría de redacción',
'Equipo editorial de Estudios de Traducción',
'TTS, LANS',
'Reviews, All',
'Sedebar, Revista',
'CLINA, Secretaría de redacción',
'de Filoloxía e Tradución, Facultade',
'editorial, Equipo ',
'Equipo Editorial Onmazein',
'Estudios de Traducción, Revista',
'de Traduccion, Estudios',
'Consejo de Redacción',
'CONSEJO EDITORIAL',
'Editorial Team, New Voices',
'New Voices Editorial Team and Guest Editors',
'Editorial Team, New Voices ',
'Issue, Complete',
'Special Issue: Intersemiotic Translation and Multimodality, Translation Matters',
'editorial, Equip',
'«Ticontre», Readazione',
'News, Transfer',
'Cret, Transfer',
'Universidad De Salamanca, Ediciones',
'Secretaría de redacción, CLINA',
'CLINA Secretaría de redacción',
'editorial, Equipo',
'Revista, Sendebar',
'Cadernos de Tradução, Cadernos',
'Varios, Autores',
'Varios, varios',
'CT, Os Editores',
'Cadernos, Editores',]

authors_stats = authors_stats[~authors_stats['creator'].isin(authors_field_to_drop)]

authors_stats = (authors_stats['creator'].value_counts())

print(authors_stats[:10])

creator
Guerini, Andréia         45
Camps, Assumpta          35
Desblache, Lucile        34
Dasilva, Xosé Manuel     32
Zieliński, Lech          32
Matamala, Anna           29
Fadini, Matteo           26
Luna Alonso, Ana         25
Costa, Walter Carlos     24
Szarkowska, Agnieszka    21
Name: count, dtype: int64


In [7]:
# authors grouping based on similarity
unique_names = authors_stats.index.tolist()

groups = []
used = set()

for name in unique_names:
    if name in used:
        continue
    
    group = [name]
    used.add(name)
    
    matches = process.extract(
        name,
        unique_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90
    )
    
    for match, score, _ in matches:
        if match not in used:
            group.append(match)
            used.add(match)
    
    groups.append(group)

authors_canonical_rows = []
for group in groups:
    canonical = max(group, key=lambda x: authors_stats[x])
    
    total_count = sum(authors_stats[n] for n in group)
    
    authors_canonical_rows.append({
        "canonical_name": canonical,
        "variants": group,
        "count": total_count
    })



In [8]:
authors_similarity_df = pd.DataFrame(authors_canonical_rows)
authors_similarity_df.head()

,canonical_name,variants,count
0,"Guerini, Andréia","[Guerini, Andréia, Guerini, Andreia]",47
1,"Camps, Assumpta","[Camps, Assumpta]",35
2,"Desblache, Lucile","[Desblache, Lucile]",34
3,"Dasilva, Xosé Manuel","[Dasilva, Xosé Manuel, Dasilva, Xose Manuel]",33
4,"Zieliński, Lech","[Zieliński, Lech]",32


In [9]:
# multiple authors stats
multiple_authors_stats = df['creator'].dropna()

multiple_authors_stats = pd.DataFrame(multiple_authors_stats)
affiliations_to_replace = ['(Chulalongkorn University, THAILAND), and ',
'(University of Manchester, UNITED KINGDOM), ',
'(Yarmouk University), ',
'(Jerash University), ',
'(Yarmouk University, JORDAN), ',
'(Beihang University, CHINA), ',
'(University of Extremadura, SPAIN), ',
'(University of Granada, SPAIN), ',
'(Universitat de València, SPAIN), ',
'(Leipzig University, GERMANY), ',
'(Guizhou Normal University, CHINA), ',
'(University Putra Malaysia, MALAYSIA; University of Newcastle, AUSTRALIA), ',
'(University Putra Malaysia, MALAYSIA), and ',
'(The John Paul II Catholic University of Lublin, POLAND), ',
'(Minia University, EGYPT), ',
'(University College London, UNITED KINGDOM), ',
'(Kent State University, UNITED STATES OF AMERICA), ',
'(The University of Western Australia, AUSTRALIA), ',
'(Queen’s University Belfast, UNITED KINGDOM), ',
'(National Police Academy, Ankara, TÜRKIYE), ',
'(Moscow City Teacher Training University, RUSSIA), ',
'(Universidad de Granada, SPAIN), ',
'(Hamad Bin Khalifa University, QATAR), ',
'(Leipzig University, GERMANY), and ',
'(University of Zadar/University of Applied Sciences Velika Gorica, CROATIA), ',
'(Kent State University, USA), ',
'(Hong Kong Baptist University, HONG KONG SAR), ',
'(Northwest Normal University/City University of Hong Kong), ',
'(City University of Hong Kong, HONG KONG SAR), ',
'(Politeknik El Bajo Commodus, INDONESIA), ',
'(Independent researcher, HONG KONG SAR), ',
', Universidad de Córdoba',]

for frag in affiliations_to_replace:
    multiple_authors_stats['creator'] = multiple_authors_stats['creator'].str.replace(frag, "", regex=False)

authors_field_to_drop = ['Tradução, Cadernos de',
'de Tradução, Cadernos',
'New Voices Editorial Team',
'Clina, Secretaría de Redacción',
'de redacción, Consello',
'Tradução, Cadernos',
'Notícias, Transfer',
'Autores, Varios',
'Transfer',
'UB, Transfer',
'Sendebar, Revista',
'Reviews, Book',
'Varios, Varios',
'Editorial, Equipo',
'de Traducción, Estudios',
'anonimo, anonimo',
'Equipo Editorial',
'Clina, Secretaría de redacción',
'Equipo editorial de Estudios de Traducción',
'TTS, LANS',
'Reviews, All',
'Sedebar, Revista',
'CLINA, Secretaría de redacción',
'de Filoloxía e Tradución, Facultade',
'editorial, Equipo ',
'Equipo Editorial Onmazein',
'Estudios de Traducción, Revista',
'de Traduccion, Estudios',
'Consejo de Redacción',
'CONSEJO EDITORIAL',
'Editorial Team, New Voices',
'New Voices Editorial Team and Guest Editors',
'Editorial Team, New Voices ',
'Issue, Complete',
'Special Issue: Intersemiotic Translation and Multimodality, Translation Matters',
'editorial, Equip',
'«Ticontre», Readazione',
'News, Transfer',
'Cret, Transfer',
'Universidad De Salamanca, Ediciones',
'Secretaría de redacción, CLINA',
'CLINA Secretaría de redacción',
'editorial, Equipo',
'Revista, Sendebar',
'Cadernos de Tradução, Cadernos',
'Varios, Autores',
'Varios, varios',
'CT, Os Editores',
'Cadernos, Editores',]

multiple_authors_stats = multiple_authors_stats[~multiple_authors_stats['creator'].isin(authors_field_to_drop)]
multiple_authors_stats = multiple_authors_stats['creator'].dropna().str.count(r'\|').add(1).value_counts()

print(multiple_authors_stats[:10])

creator
1     7094
2     1476
3      408
4       95
5       41
6       19
7        6
8        3
9        1
22       1
Name: count, dtype: int64


In [10]:
# language stats
lang_stats = (df['language']
                .dropna()
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .str.replace(r'^en$', 'eng', regex=True)
                .str.replace(r'^es$', 'spa', regex=True)
                .dropna()
                .value_counts())

print(lang_stats[:10])

language
spa    3927
eng    3183
por    1565
pol     381
ita     317
cat     115
fra      46
deu      13
Name: count, dtype: int64


In [11]:
# dates stats
dates_stats = (df['date']
        .dropna()
        .str.split(' | ', regex=False, expand=False) 
        .explode()
        .dropna()
        .str.replace(r'-.+$', '', regex=True)
        .value_counts())
print(dates_stats[:10])

earliest = df['date'].min()
latest = df['date'].max()
print(earliest, latest)

df_years_kwds = df.copy()
df_years_kwds["year"] = pd.to_datetime(df_years_kwds["date"]).dt.year
df_years_kwds["subject"] = (df_years_kwds["subject"]
                                .str.replace('; ', ' | ', regex=False)
                                .str.replace('；', ' | ', regex=False)
                                .str.replace(', ', ' | ', regex=False)
                                .str.replace('，', ' | ', regex=False)
                                .str.split(' | ', regex=False, expand=False))
df_years_kwds = df_years_kwds.explode("subject")
df_years_kwds["subject"] = (
    df_years_kwds["subject"]
    .str.strip('| .,;')
    .str.lower()
)

with open('translated_keywords.txt', 'r', encoding='utf-8') as txt:
    keywords_mapping = [e.strip() for e in txt.readlines()]
keywords_mapping = {e.split(' [--translated-->] ')[0]:e.split(' [--translated-->] ')[1] for e in keywords_mapping}
df_years_kwds["subject"] = df_years_kwds["subject"].replace(keywords_mapping).str.lower().str.strip()
df_years_kwds = df_years_kwds.drop_duplicates(subset=["oai_id", "subject"], keep="first")
df_years_kwds_grouped = df_years_kwds.dropna(subset=["subject"]).groupby("year")["subject"].value_counts()

print(df_years_kwds_grouped.head(10))

date
2023    1130
2020     910
2021     859
2017     772
2022     641
2024     589
2019     503
2018     490
2016     372
2025     357
Name: count, dtype: int64
1996-01-01 2025-10-24
year  subject             
1999  ascriptive sentence     1
      attribution             1
      attributive sentence    1
      be and be               1
      equative sentence       1
2002  faithfulness            2
      fidelity                2
      aesthetics              1
      arabic                  1
      blazon                  1
Name: count, dtype: int64


In [12]:
with pd.ExcelWriter('metadata_stats_v0.4.xlsx') as writer:
    keywords_stats_multilang.reset_index().rename(columns={'index':'keyword'}).to_excel(writer, sheet_name='keywords_stats_multilang', index=False)
    keywords_stats_en.reset_index().rename(columns={'index':'keyword'}).to_excel(writer, sheet_name='keywords_stats_en', index=False)
    keywords_grouped.to_excel(writer, sheet_name='keywords_grouped', index=False)
    authors_stats.reset_index().rename(columns={'index':'author'}).to_excel(writer, sheet_name='authors_stats', index=False)
    authors_similarity_df.to_excel(writer, sheet_name='authors_similarity', index=False)
    multiple_authors_stats.reset_index().rename(columns={'index':'authors_number'}).to_excel(writer, sheet_name='multiple_authors_stats', index=False)
    lang_stats.reset_index().rename(columns={'index':'language'}).to_excel(writer, sheet_name='lang_stats', index=False)
    df_years_kwds_grouped.reset_index().rename(columns={'index':'year'}).to_excel(writer, sheet_name='years_kwds_grouped', index=False)